In [0]:
import urllib.request
import csv
import io

GITHUB_BASE = "https://raw.githubusercontent.com/AparnaAmonkar22/shell-energy-uk-risk-analytics/main/data-generation/dev_subset/"

def fetch_csv_as_dicts(filename):
    url = GITHUB_BASE + filename
    with urllib.request.urlopen(url) as response:
        text = response.read().decode("utf-8")
    return list(csv.DictReader(io.StringIO(text)))

sites_rows = fetch_csv_as_dicts("sites.csv")
customers_rows = fetch_csv_as_dicts("customers.csv")

print(f"Fetched {len(sites_rows)} sites, {len(customers_rows)} customers from GitHub.")

sector_by_customer = {row["customer_id"]: row["sector"] for row in customers_rows}

site_metadata = {
    row["site_id"]: {
        "mpan": row["mpan"],
        "annual_baseline_kwh": float(row["annual_baseline_kwh"]),
        "sector": sector_by_customer.get(row["customer_id"], "Office / Commercial Real Estate"),
    }
    for row in sites_rows
}

print(f"Built metadata for {len(site_metadata)} sites.")

In [0]:
%pip install azure-eventhub

In [0]:
import math
import random
import json
import time
from datetime import datetime, timedelta, timezone

random.seed()

def sector_load_factor(sector, dt):
    hour = dt.hour + dt.minute / 60.0
    is_weekend = dt.weekday() >= 5
    if sector == "Office / Commercial Real Estate":
        return 0.08 if is_weekend else 0.05 + 0.95 * math.exp(-((hour - 13) ** 2) / (2 * 4.0 ** 2))
    if sector == "Data Centre":
        return 0.92 + 0.08 * math.sin((hour / 24) * 2 * math.pi)
    if sector == "Retail":
        return 0.25 + 0.75 * math.exp(-((hour - 14) ** 2) / (2 * 5.5 ** 2))
    if sector == "Manufacturing":
        return 0.75 if 6 <= hour < 22 and not is_weekend else 0.25
    return 0.5

def gen_meter_event(site_id, meta, sim_time):
    shape = sector_load_factor(meta["sector"], sim_time)
    avg_interval_kwh = meta["annual_baseline_kwh"] / 17520.0
    noise = random.gauss(1.0, 0.08)
    kwh = max(avg_interval_kwh * shape * noise * 2.1, 0)

    meter_status = "Normal"
    if random.random() < 0.002:
        meter_status = "Fault"
        kwh = 0.0

    return {
        "event_type": "meter_reading",
        "site_id": site_id,
        "mpan": meta["mpan"],
        "interval_start": sim_time.isoformat(),
        "kwh_interval": round(kwh, 4),
        "meter_status": meter_status,
        "ingested_at": datetime.now(timezone.utc).isoformat(),
    }

def gen_price_event(sim_time):
    hour = sim_time.hour + sim_time.minute / 60.0
    morning = 0.9 * math.exp(-((hour - 8.0) ** 2) / (2 * 2.2 ** 2))
    evening = 1.3 * math.exp(-((hour - 18.0) ** 2) / (2 * 2.5 ** 2))
    shape = 0.35 + morning + evening
    price = max(60.0 * (1 + shape) + random.gauss(0, 5), 5.0)
    return {
        "event_type": "wholesale_price",
        "interval_start": sim_time.isoformat(),
        "price_gbp_per_mwh": round(price, 2),
        "market_period": "Peak" if 7 <= hour < 23 else "Off-Peak",
        "ingested_at": datetime.now(timezone.utc).isoformat(),
    }

print("Event generation functions defined.")

In [0]:
eventhub_connection_string = dbutils.secrets.get(
    scope="aparna-kv-shellenergy-scope",
    key="aparna-eventhub-connection-string"
)

print("Event Hub connection string retrieved from Key Vault.")

In [0]:
from azure.eventhub import EventHubProducerClient, EventData

meter_eventhub_name = "aparna-meter-readings-stream"
price_eventhub_name = "aparna-wholesale-price-stream"

meter_producer = EventHubProducerClient.from_connection_string(
    conn_str=eventhub_connection_string, eventhub_name=meter_eventhub_name
)
price_producer = EventHubProducerClient.from_connection_string(
    conn_str=eventhub_connection_string, eventhub_name=price_eventhub_name
)

print("Producer clients created for both Event Hubs.")

In [0]:
SIM_SECONDS_PER_INTERVAL = 5
SITES_SAMPLE = 5

site_ids = list(site_metadata.keys())
sample = random.sample(site_ids, min(SITES_SAMPLE, len(site_ids)))

sim_time = datetime.now(timezone.utc).replace(minute=0, second=0, microsecond=0)

print(f"Starting producer loop. {len(sample)} sites. Interrupt this cell (Cancel) to stop.")

try:
    while True:
        price_event = gen_price_event(sim_time)
        meter_events = [gen_meter_event(site_id, site_metadata[site_id], sim_time) for site_id in sample]

        price_batch = price_producer.create_batch()
        price_batch.add(EventData(json.dumps(price_event)))
        price_producer.send_batch(price_batch)

        meter_batch = meter_producer.create_batch()
        for e in meter_events:
            meter_batch.add(EventData(json.dumps(e)))
        meter_producer.send_batch(meter_batch)

        print(f"Sent 1 price event + {len(meter_events)} meter events for sim_time={sim_time.isoformat()}")

        sim_time += timedelta(minutes=30)
        time.sleep(SIM_SECONDS_PER_INTERVAL)
except KeyboardInterrupt:
    print("Stopped.")
finally:
    meter_producer.close()
    price_producer.close()